# 14 · Scale and Robustness of Descriptor-Space Universality

This notebook tests whether the main geometric result from Notebooks 10–13 is **stable under representation choices**.

The main question is:

> does zeta remain closer to GUE than to Poisson when we vary:
> - window length
> - sample size
> - descriptor set

## What this notebook does

We compare descriptor-space distances for several settings:

1. window lengths: 3-gap, 5-gap, 7-gap
2. sample sizes: smaller and larger zeta segments
3. descriptor sets:
   - **full** descriptor set
   - **shape-only** descriptor set
   - **summary-only** descriptor set

For each setting, we compute:
- zeta–GUE distance
- zeta–Poisson distance
- the gap:
  \[
  d(\text{zeta}, \text{Poisson}) - d(\text{zeta}, \text{GUE})
  \]

The goal is not to optimize any one representation, but to test whether the closeness result is robust.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
# Larger zeta sample so we can test smaller/larger prefixes
N = 1000
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

# Poisson control matched to full zeta length
poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

# Finite GUE bulk gaps
def sample_gue_bulk_spacings(matrix_size=140, n_matrices=50, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep
        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=140, n_matrices=50, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window and descriptor helpers

In [ ]:
def extract_windows(gaps, k):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_features(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r)), float(np.std(r))

def descriptor_vector(window, mode="full"):
    ratio_mean, ratio_std = ratio_features(window)

    if mode == "full":
        return np.array(
            list(window) +
            [unevenness(window), reversal_symmetry_score(window), window_entropy(window), ratio_mean, ratio_std],
            dtype=float
        )
    elif mode == "shape_only":
        return np.array(list(window), dtype=float)
    elif mode == "summary_only":
        return np.array(
            [unevenness(window), reversal_symmetry_score(window), window_entropy(window), ratio_mean, ratio_std],
            dtype=float
        )
    else:
        raise ValueError(f"Unknown mode: {mode}")

def build_descriptors(gaps, k=5, mode="full"):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    desc = np.array([descriptor_vector(w, mode=mode) for w in windows_n])
    return windows_n, desc

## PCA + distance helpers

In [ ]:
def pca_embed_three(A, B, C):
    all_desc = np.vstack([A, B, C])
    mean = all_desc.mean(axis=0)
    std = all_desc.std(axis=0)
    std[std == 0] = 1.0

    A_std = (A - mean) / std
    B_std = (B - mean) / std
    C_std = (C - mean) / std

    X = np.vstack([A_std, B_std, C_std])
    Xc = X - X.mean(axis=0)

    cov = np.cov(Xc, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    eigvecs = eigvecs[:, order]
    comps = eigvecs[:, :2]

    emb = Xc @ comps
    nA, nB, nC = len(A_std), len(B_std), len(C_std)

    return emb[:nA], emb[nA:nA+nB], emb[nA+nB:]

def centroid_distance(A, B):
    return float(np.linalg.norm(A.mean(axis=0) - B.mean(axis=0)))

def density_grid(X, bins_x, bins_y):
    H, _, _ = np.histogram2d(X[:, 0], X[:, 1], bins=[bins_x, bins_y], density=True)
    return H

def grid_l2_distance(H1, H2):
    return float(np.linalg.norm(H1.ravel() - H2.ravel()))

def compute_geometry_metrics(zeta_desc, poisson_desc, gue_desc):
    zeta_emb, poisson_emb, gue_emb = pca_embed_three(zeta_desc, poisson_desc, gue_desc)

    xmin = min(zeta_emb[:,0].min(), poisson_emb[:,0].min(), gue_emb[:,0].min())
    xmax = max(zeta_emb[:,0].max(), poisson_emb[:,0].max(), gue_emb[:,0].max())
    ymin = min(zeta_emb[:,1].min(), poisson_emb[:,1].min(), gue_emb[:,1].min())
    ymax = max(zeta_emb[:,1].max(), poisson_emb[:,1].max(), gue_emb[:,1].max())

    bins_x = np.linspace(xmin, xmax, 30)
    bins_y = np.linspace(ymin, ymax, 30)

    zeta_H = density_grid(zeta_emb, bins_x, bins_y)
    poisson_H = density_grid(poisson_emb, bins_x, bins_y)
    gue_H = density_grid(gue_emb, bins_x, bins_y)

    cd_zg = centroid_distance(zeta_emb, gue_emb)
    cd_zp = centroid_distance(zeta_emb, poisson_emb)

    gd_zg = grid_l2_distance(zeta_H, gue_H)
    gd_zp = grid_l2_distance(zeta_H, poisson_H)

    return {
        "centroid_zeta_GUE": cd_zg,
        "centroid_zeta_Poisson": cd_zp,
        "centroid_gap": cd_zp - cd_zg,
        "grid_zeta_GUE": gd_zg,
        "grid_zeta_Poisson": gd_zp,
        "grid_gap": gd_zp - gd_zg,
        "zeta_emb": zeta_emb,
        "poisson_emb": poisson_emb,
        "gue_emb": gue_emb,
    }

## Robustness sweep

We vary:
- sample size: first 300, 600, 900 zeta gaps
- window length: 3, 5, 7
- descriptor set: full, shape_only, summary_only

In [ ]:
sample_sizes = [300, 600, 900]
window_lengths = [3, 5, 7]
descriptor_modes = ["full", "shape_only", "summary_only"]

results = []

for n in sample_sizes:
    zeta_gaps = zeta_gaps_full[:n]
    poisson_gaps = poisson_gaps_full[:n]

    # match GUE by taking a prefix with enough length
    gue_gaps = gue_gaps_full[:max(n, 950)]

    for k in window_lengths:
        for mode in descriptor_modes:
            _, zeta_desc = build_descriptors(zeta_gaps, k=k, mode=mode)
            _, poisson_desc = build_descriptors(poisson_gaps, k=k, mode=mode)
            _, gue_desc = build_descriptors(gue_gaps, k=k, mode=mode)

            # downsample to common size for fair comparison
            m = min(len(zeta_desc), len(poisson_desc), len(gue_desc))
            zeta_desc_m = zeta_desc[:m]
            poisson_desc_m = poisson_desc[:m]
            gue_desc_m = gue_desc[:m]

            metrics = compute_geometry_metrics(zeta_desc_m, poisson_desc_m, gue_desc_m)

            results.append({
                "sample_size": n,
                "window_length": k,
                "descriptor_mode": mode,
                "centroid_zeta_GUE": metrics["centroid_zeta_GUE"],
                "centroid_zeta_Poisson": metrics["centroid_zeta_Poisson"],
                "centroid_gap": metrics["centroid_gap"],
                "grid_zeta_GUE": metrics["grid_zeta_GUE"],
                "grid_zeta_Poisson": metrics["grid_zeta_Poisson"],
                "grid_gap": metrics["grid_gap"],
            })

len(results)

## Raw robustness results

In [ ]:
results[:5]

## Plot centroid-gap robustness

Positive values mean:
\[
d(\text{zeta}, \text{Poisson}) > d(\text{zeta}, \text{GUE})
\]
so larger is stronger.

In [ ]:
labels = []
centroid_gaps = []

for r in results:
    labels.append(f"n={r['sample_size']}, k={r['window_length']}, {r['descriptor_mode']}")
    centroid_gaps.append(r["centroid_gap"])

plt.figure(figsize=(14, 5.2))
plt.bar(np.arange(len(labels)), centroid_gaps)
plt.xticks(np.arange(len(labels)), labels, rotation=90)
plt.ylabel("centroid gap")
plt.title("Robustness of centroid gap across settings")
plt.tight_layout()
plt.show()

## Plot grid-gap robustness

In [ ]:
grid_gaps = [r["grid_gap"] for r in results]

plt.figure(figsize=(14, 5.2))
plt.bar(np.arange(len(labels)), grid_gaps)
plt.xticks(np.arange(len(labels)), labels, rotation=90)
plt.ylabel("grid L2 gap")
plt.title("Robustness of density-grid gap across settings")
plt.tight_layout()
plt.show()

## Summaries grouped by descriptor mode

In [ ]:
grouped = {}
for mode in descriptor_modes:
    mode_rows = [r for r in results if r["descriptor_mode"] == mode]
    grouped[mode] = {
        "centroid_gap_mean": float(np.mean([r["centroid_gap"] for r in mode_rows])),
        "centroid_gap_min": float(np.min([r["centroid_gap"] for r in mode_rows])),
        "grid_gap_mean": float(np.mean([r["grid_gap"] for r in mode_rows])),
        "grid_gap_min": float(np.min([r["grid_gap"] for r in mode_rows])),
    }

grouped

## Summaries grouped by window length

In [ ]:
by_k = {}
for k in window_lengths:
    rows = [r for r in results if r["window_length"] == k]
    by_k[k] = {
        "centroid_gap_mean": float(np.mean([r["centroid_gap"] for r in rows])),
        "centroid_gap_min": float(np.min([r["centroid_gap"] for r in rows])),
        "grid_gap_mean": float(np.mean([r["grid_gap"] for r in rows])),
        "grid_gap_min": float(np.min([r["grid_gap"] for r in rows])),
    }

by_k

## Summaries grouped by sample size

In [ ]:
by_n = {}
for n in sample_sizes:
    rows = [r for r in results if r["sample_size"] == n]
    by_n[n] = {
        "centroid_gap_mean": float(np.mean([r["centroid_gap"] for r in rows])),
        "centroid_gap_min": float(np.min([r["centroid_gap"] for r in rows])),
        "grid_gap_mean": float(np.mean([r["grid_gap"] for r in rows])),
        "grid_gap_min": float(np.min([r["grid_gap"] for r in rows])),
    }

by_n

## Representative PCA embeddings

Show one embedding for each descriptor mode at fixed:
- sample size = 600
- window length = 5

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

for ax, mode in zip(axes, descriptor_modes):
    zeta_gaps = zeta_gaps_full[:600]
    poisson_gaps = poisson_gaps_full[:600]
    gue_gaps = gue_gaps_full[:950]

    _, zeta_desc = build_descriptors(zeta_gaps, k=5, mode=mode)
    _, poisson_desc = build_descriptors(poisson_gaps, k=5, mode=mode)
    _, gue_desc = build_descriptors(gue_gaps, k=5, mode=mode)

    m = min(len(zeta_desc), len(poisson_desc), len(gue_desc))
    zeta_emb, poisson_emb, gue_emb = pca_embed_three(zeta_desc[:m], poisson_desc[:m], gue_desc[:m])

    ax.scatter(zeta_emb[:, 0], zeta_emb[:, 1], s=8, alpha=0.22, label="zeta")
    ax.scatter(poisson_emb[:, 0], poisson_emb[:, 1], s=8, alpha=0.22, label="Poisson")
    ax.scatter(gue_emb[:, 0], gue_emb[:, 1], s=8, alpha=0.22, label="GUE")
    ax.set_title(mode)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

axes[0].legend()
plt.tight_layout()
plt.show()

## Final summary

In [ ]:
summary = {
    "grouped_by_descriptor_mode": grouped,
    "grouped_by_window_length": by_k,
    "grouped_by_sample_size": by_n,
    "all_results": results,
}
summary

## Notes

- The main robustness criterion is whether the zeta–Poisson distance remains larger than the zeta–GUE distance across many settings.
- If the gap stays positive for different window lengths, sample sizes, and descriptor sets, then the universality signal is not a fragile artifact of one representation choice.
- The "shape_only" mode tests whether the raw normalized window coordinates already carry the signal.
- The "summary_only" mode tests whether a low-dimensional handcrafted descriptor set is enough.

## Next directions

A natural next notebook could do one of these:

1. bootstrap these robustness sweeps for uncertainty bars on every setting  
2. add conditional-window robustness (after small vs after large gaps)  
3. compare alternative embeddings beyond PCA  
4. test whether the same geometry story appears in other spectral datasets